<a href="https://colab.research.google.com/github/HemanthSagarSR/BIS_LAB/blob/main/week3_ACO_hospital_resource_allocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np

class HospitalACO:
    def __init__(self, patient_demands, resource_capacities, patient_priorities,
                 allocation_costs, num_ants=10, max_iter=50, alpha=1.0, beta=2.0, rho=0.1):
        self.demands = np.array(patient_demands)
        self.capacities = np.array(resource_capacities)
        self.priorities = np.array(patient_priorities)
        self.costs = np.array(allocation_costs)
        self.num_patients = len(patient_demands)
        self.num_resources = len(resource_capacities)
        self.num_ants = num_ants
        self.max_iter = max_iter
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.pheromones = np.ones((self.num_patients, self.num_resources))

    def _calculate_heuristic(self):
        heuristic = np.zeros((self.num_patients, self.num_resources))
        for p in range(self.num_patients):
            for r in range(self.num_resources):
                waiting_time_estimate = 1.0 / (1.0 + self.capacities[r] - self.demands[p])
                base_score = (self.priorities[p] / (self.costs[p][r] + 1e-5)) * waiting_time_estimate
                heuristic[p][r] = max(base_score, 1e-5)
        return heuristic

    def solve(self):
        best_allocation = None
        best_score = float('inf')
        heuristic = self._calculate_heuristic()

        for _ in range(self.max_iter):
            ant_allocations = []
            ant_scores = []

            for _ in range(self.num_ants):
                allocation = np.full(self.num_patients, -1, dtype=int)
                current_capacities = self.capacities.copy()
                valid_schedule = True
                sorted_patients = np.argsort(-self.priorities)

                for p in sorted_patients:
                    probabilities = []
                    feasible_resources = []

                    for r in range(self.num_resources):
                        if current_capacities[r] >= self.demands[p]:
                            tau = self.pheromones[p][r] ** self.alpha
                            eta = heuristic[p][r] ** self.beta
                            probabilities.append(tau * eta)
                            feasible_resources.append(r)

                    if not probabilities:
                        valid_schedule = False
                        break

                    prob_sum = sum(probabilities)
                    probabilities = [prob / prob_sum for prob in probabilities]
                    chosen_resource = np.random.choice(feasible_resources, p=probabilities)
                    allocation[p] = chosen_resource
                    current_capacities[chosen_resource] -= self.demands[p]

                if valid_schedule:
                    score = 0
                    for p, r in enumerate(allocation):
                        score += self.costs[p][r] * (4 - self.priorities[p])
                    ant_allocations.append(allocation)
                    ant_scores.append(score)

            self.pheromones *= (1 - self.rho)

            for idx, allocation in enumerate(ant_allocations):
                score = ant_scores[idx]
                if score < best_score:
                    best_score = score
                    best_allocation = allocation.copy()

                for p, r in enumerate(allocation):
                    self.pheromones[p][r] += (1.0 / (score + 1e-5))

        return best_allocation, best_score

if __name__ == "__main__":
    num_patients = int(input("Enter number of patients: "))
    num_resources = int(input("Enter number of resource types: "))

    print(f"Enter resource demands for the {num_patients} patients (space-separated):")
    patients = [int(x) for x in input().split()]

    print(f"Enter total capacities for the {num_resources} resources (space-separated):")
    resources = [int(x) for x in input().split()]

    print(f"Enter priority levels (e.g., 1-3) for the {num_patients} patients (space-separated):")
    priorities = [int(x) for x in input().split()]

    print(f"Enter the cost matrix row by row (each row needs {num_resources} space-separated allocation costs):")
    costs = []
    for i in range(num_patients):
        row = [float(x) for x in input(f"Patient {i} costs: ").split()]
        costs.append(row)

    aco = HospitalACO(patients, resources, priorities, costs)
    schedule, total_cost = aco.solve()

    print("\n--- Optimization Complete ---")
    print("Allocation Schedule:", schedule)
    print("Minimized Cost Score:", total_cost)


Enter number of patients: 3
Enter number of resource types: 2
Enter resource demands for the 3 patients (space-separated):
2 1 3
Enter total capacities for the 2 resources (space-separated):
4 3
Enter priority levels (e.g., 1-3) for the 3 patients (space-separated):
3 1 2
Enter the cost matrix row by row (each row needs 2 space-separated allocation costs):
Patient 0 costs: 10 25
Patient 1 costs: 15 5
Patient 2 costs: 30 12

--- Optimization Complete ---
Allocation Schedule: [0 0 1]
Minimized Cost Score: 79.0
